# Tracklet Velocity: Benchmark v3 vs Sorcha cases on MJD 61642

Reproduces the scatter + quiver analysis from `tracklet_comparison_sorcha_vs_benchmark.ipynb`
using:
- **Benchmark v3** — proportional population caps (MBA 96.5%, NEO 1.9%, Trojan 1.25%, TNO 0.34%)
  + epoch MJD **61642** (2027-08-25), the busiest single night in every Sorcha case.
- **Sorcha case1 / case2 / case3** — all filtered to night MJD 61642.

**Matching:** each Sorcha ObjID is looked up in the S3M files to get its V-band (e, H);
those are matched against benchmark v3's (e, H) via nearest-neighbour with a tight
tolerance of 1 × 10⁻⁴ in (e, H) space.  Only unique, unambiguous pairs are kept.

**Cross-case comparison (Section 5):** 663 NEO ObjIDs appear on MJD 61642 in all three
Sorcha cases — exactly the same objects, three different linking configurations.  This lets
us ask whether SSP linking changes the *measured* (vλ, vβ) for the same NEO.

In [ ]:
import glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from scipy.spatial import cKDTree

NIGHT   = 61642
MATCH_TOL = 1e-4          # (e, H) distance threshold for a confident match
POP_COLORS = {'NEO': 'tab:red', 'MBA': 'tab:blue', 'TNO': 'gold',
               'Trojan': 'mediumseagreen', 'other': 'grey'}
CASE_LS  = {'case1': '-',  'case2': '--', 'case3': ':'}
CASE_LAB = {
    'case1': 'case1 (tk=1, sep=0.5")',
    'case2': 'case2 (tk=3, sep=0.5" — real LSST)',
    'case3': 'case3 (tk=1, sep≈0, baseline)',
}

In [ ]:
# ── parse S3M files → (ObjID, e, H_V) lookup ─────────────────────────────────
# col index: 0=ObjID, 3=e, 8=H  (same as original tracklet_comparison notebook)
records = []
for f in sorted(glob.glob('S*.s3m')):
    with open(f) as fh:
        for line in fh:
            if line.startswith('!'): continue
            p = line.split()
            if len(p) < 9: continue
            try: records.append((p[0], float(p[3]), float(p[8])))
            except ValueError: pass
s3m_eh = (pd.DataFrame(records, columns=['ObjID','e_s3m','H_s3m'])
          .drop_duplicates('ObjID').set_index('ObjID'))
print(f'S3M lookup: {len(s3m_eh):,} objects')

In [ ]:
# ── load benchmark v3 (mag_bin_label filtered = VDP-scored subset) ─────────────
bv3_raw = pd.read_parquet('docs/benchmark_comparison_s3m_v3.parquet')
bv3 = bv3_raw[bv3_raw.mag_bin_label.notna()].reset_index(drop=True).copy()
bv3['absdlon'] = bv3.dlon_from_antisun_deg.abs()
tree = cKDTree(bv3[['e','H']].values)
print(f'Benchmark v3 (scored): {len(bv3):,} rows')
print(f'  pops: {dict(bv3.population.value_counts())}')

# ── load Sorcha cases, filter to busiest night ────────────────────────────────
cases = {}
for c in ['case1','case2','case3']:
    df = pd.read_parquet(f'docs/sorcha_comparison_{c}.parquet')
    night = df[df.night == NIGHT].copy()
    night = night.join(s3m_eh, on='ObjID').dropna(subset=['e_s3m','H_s3m'])
    cases[c] = night
    print(f'{c} night {NIGHT}: {len(night)} tracklets — {dict(night.population.value_counts())}')

In [ ]:
# ── match each case → benchmark v3 ───────────────────────────────────────────
matched = {}
for c, night in cases.items():
    dist, idx = tree.query(night[['e_s3m','H_s3m']].values, k=1)
    m = dist < MATCH_TOL
    hit = night[m].copy()
    hit_idx = idx[m]
    hit['vlam_b']  = bv3.vlam.values[hit_idx]
    hit['vbeta_b'] = bv3.vbeta.values[hit_idx]
    hit['pop_b']   = bv3.population.values[hit_idx]
    hit['dv']      = np.hypot(hit.vlam - hit.vlam_b, hit.vbeta - hit.vbeta_b)
    matched[c] = hit
    print(f'{c}: {len(hit)} matched pairs  |  pops: {dict(hit.population.value_counts())}')
    print(f'  |Δv| median={hit.dv.median():.3f}  mean={hit.dv.mean():.3f} deg/day')

## Section 1 — Night 61642 Population Summary

In [ ]:
rows = []
for c, night in cases.items():
    vc = night.population.value_counts()
    row = {'Case': c, 'Total tracklets': len(night)}
    for p in ['NEO','MBA','TNO','Trojan','other']:
        n = int(vc.get(p,0))
        row[p] = f'{n} ({100*n/len(night):.1f}%)'
    row['Matched to v3'] = len(matched[c])
    rows.append(row)
print(f'Night MJD {NIGHT} — population breakdown')
display(pd.DataFrame(rows).set_index('Case'))

## Section 2 — Scatter Plot: Benchmark v3 vs Sorcha (all matched pairs)

Reproduces the first image.  Blue triangles = benchmark v3, red circles = Sorcha.  
Gray lines connect the same object across the two pipelines.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), sharex=True, sharey=True)

for ax, (c, hit) in zip(axes, matched.items()):
    # gray connector lines
    for _, row in hit.iterrows():
        ax.plot([row.vlam_b, row.vlam], [row.vbeta_b, row.vbeta],
                color='gray', lw=0.5, alpha=0.4, zorder=1)

    # benchmark points (triangles)
    ax.scatter(hit.vlam_b, hit.vbeta_b, marker='^', s=30, alpha=0.7,
               color='tab:blue', label=f'Benchmark v3 (n={len(hit)})', zorder=3)

    # sorcha points (circles), coloured by population
    for pop, grp in hit.groupby('population'):
        ax.scatter(grp.vlam, grp.vbeta, marker='o', s=25, alpha=0.75,
                   color=POP_COLORS.get(pop,'grey'), label=f'Sorcha {pop} (n={len(grp)})', zorder=4)

    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel(r'$v_\lambda$ (deg/day)')
    ax.legend(fontsize=7, loc='upper right')
    ax.set_title(f'{CASE_LAB[c]}\n{len(hit)} matched pairs — MJD {NIGHT}', fontsize=9)
    ax.grid(alpha=0.2)

axes[0].set_ylabel(r'$v_\beta$ (deg/day)')
fig.suptitle(f'Ecliptic rate space — Benchmark v3 vs Sorcha (MJD {NIGHT})', fontsize=11)
plt.tight_layout()
plt.savefig('Figures/tracklet_scatter_v3_cases.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 3 — Quiver Plot: Velocity Vectors from Origin

Reproduces the second image.  Each arrow = one matched pair.  
Blue = benchmark v3, red = Sorcha.  Pairs are layered so mismatches are visible.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), sharex=True, sharey=True)

for ax, (c, hit) in zip(axes, matched.items()):
    zeros = np.zeros(len(hit))
    ax.quiver(zeros, zeros, hit.vlam_b.values, hit.vbeta_b.values,
              color='tab:blue', alpha=0.45, angles='xy', scale_units='xy', scale=1,
              width=0.004, label='Benchmark v3')
    ax.quiver(zeros, zeros, hit.vlam.values,   hit.vbeta.values,
              color='tab:red',  alpha=0.45, angles='xy', scale_units='xy', scale=1,
              width=0.004, label='Sorcha')
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel(r'$v_\lambda$ (deg/day)')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_title(f'{CASE_LAB[c]}\n{len(hit)} pairs  |Δv| med={hit.dv.median():.2f} deg/day', fontsize=9)
    ax.grid(alpha=0.2)

axes[0].set_ylabel(r'$v_\beta$ (deg/day)')
fig.suptitle(f'Velocity vectors from origin — Benchmark v3 vs Sorcha (MJD {NIGHT})', fontsize=11)
plt.tight_layout()
plt.savefig('Figures/tracklet_quiver_v3_cases.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 4 — NEO-Only Focus

Zoom in on the NEO matched pairs — the scientifically most important population.  
Scatter + quiver side by side for each case.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10), sharex=True, sharey=True)

for col, (c, hit) in enumerate(matched.items()):
    neo = hit[hit.population == 'NEO']
    n   = len(neo)

    # top row: scatter
    ax = axes[0, col]
    for _, row in neo.iterrows():
        ax.plot([row.vlam_b, row.vlam], [row.vbeta_b, row.vbeta],
                color='gray', lw=0.8, alpha=0.5, zorder=1)
    ax.scatter(neo.vlam_b, neo.vbeta_b, marker='^', s=55, alpha=0.85,
               color='tab:blue', label=f'Benchmark v3 (n={n})', zorder=3)
    ax.scatter(neo.vlam,   neo.vbeta,   marker='o', s=45, alpha=0.85,
               color='tab:red',  label=f'Sorcha (n={n})',        zorder=4)
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_title(f'{CASE_LAB[c]} — NEOs only', fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.25)

    # bottom row: quiver
    ax = axes[1, col]
    zeros = np.zeros(n)
    ax.quiver(zeros, zeros, neo.vlam_b.values, neo.vbeta_b.values,
              color='tab:blue', alpha=0.6, angles='xy', scale_units='xy', scale=1,
              width=0.006, label='Benchmark v3')
    ax.quiver(zeros, zeros, neo.vlam.values,   neo.vbeta.values,
              color='tab:red',  alpha=0.6, angles='xy', scale_units='xy', scale=1,
              width=0.006, label='Sorcha')
    dv_neo = neo.dv
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel(r'$v_\lambda$ (deg/day)')
    ax.legend(fontsize=7)
    ax.set_title(f'|Δv| med={dv_neo.median():.2f}  mean={dv_neo.mean():.2f} deg/day', fontsize=8)
    ax.grid(alpha=0.25)

axes[0,0].set_ylabel(r'Scatter — $v_\beta$ (deg/day)')
axes[1,0].set_ylabel(r'Quiver — $v_\beta$ (deg/day)')
fig.suptitle(f'NEO tracklets only — Benchmark v3 vs Sorcha cases (MJD {NIGHT})', fontsize=11)
plt.tight_layout()
plt.savefig('Figures/tracklet_neo_v3_cases.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── |Δv| stats table for NEOs ────────────────────────────────────────────────
rows = []
for c, hit in matched.items():
    neo = hit[hit.population == 'NEO']
    rows.append({
        'Case': c,
        'N_NEO pairs': len(neo),
        '|Δv| median': round(neo.dv.median(), 3),
        '|Δv| mean':   round(neo.dv.mean(),   3),
        '|Δv| p90':    round(neo.dv.quantile(0.9), 3),
        '|Δv| max':    round(neo.dv.max(), 3),
    })
print('=== NEO velocity offset: Sorcha vs Benchmark v3 ===')
display(pd.DataFrame(rows).set_index('Case'))

## Section 5 — Cross-Case Velocity Comparison (same 663 NEOs)

663 NEO ObjIDs appear on MJD 61642 in **all three** Sorcha cases.  
This is a pure same-object, different-linking-configuration test:  
does the SSP linking choice change the measured (vλ, vβ) for the same NEO?

In [ ]:
# ── build common-NEO table ────────────────────────────────────────────────────
nights_neo = {}
for c in ['case1','case2','case3']:
    df = pd.read_parquet(f'docs/sorcha_comparison_{c}.parquet',
                         columns=['ObjID','night','population','vlam','vbeta',
                                  'P_NEO_vdp','P_NEO_d2','dt_min','mean_mag'])
    nights_neo[c] = df[(df.night == NIGHT) & (df.population == 'NEO')].set_index('ObjID')

common_ids = set(nights_neo['case1'].index) & set(nights_neo['case2'].index) & set(nights_neo['case3'].index)
print(f'NEOs common to all 3 cases on night {NIGHT}: {len(common_ids)}')

# align
ids = sorted(common_ids)
c1n = nights_neo['case1'].loc[ids]
c2n = nights_neo['case2'].loc[ids]
c3n = nights_neo['case3'].loc[ids]

dv12 = np.hypot(c1n.vlam - c2n.vlam, c1n.vbeta - c2n.vbeta)
dv13 = np.hypot(c1n.vlam - c3n.vlam, c1n.vbeta - c3n.vbeta)
dv23 = np.hypot(c2n.vlam - c3n.vlam, c2n.vbeta - c3n.vbeta)
print(f'|Δv| case1 vs case2: median={dv12.median():.4f}  mean={dv12.mean():.4f} deg/day')
print(f'|Δv| case1 vs case3: median={dv13.median():.4f}  mean={dv13.mean():.4f} deg/day')
print(f'|Δv| case2 vs case3: median={dv23.median():.4f}  mean={dv23.mean():.4f} deg/day')

In [ ]:
# ── cross-case quiver: case1 (blue) vs case2 (orange) vs case3 (green) ────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# left: quiver overlay (all from origin)
ax = axes[0]
zeros = np.zeros(len(ids))
colors = {'case1': 'tab:blue', 'case2': 'tab:orange', 'case3': 'tab:green'}
for c, nn in [('case3', c3n), ('case1', c1n), ('case2', c2n)]:
    ax.quiver(zeros, zeros, nn.vlam.values, nn.vbeta.values,
              color=colors[c], alpha=0.4, angles='xy', scale_units='xy', scale=1,
              width=0.004, label=CASE_LAB[c])
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_xlabel(r'$v_\lambda$ (deg/day)'); ax.set_ylabel(r'$v_\beta$ (deg/day)')
ax.legend(fontsize=8); ax.grid(alpha=0.2)
ax.set_title(f'Same 663 NEOs — velocity quiver overlay')

# right: case1 vs case2 scatter (direct comparison)
ax = axes[1]
ax.scatter(c1n.vlam, c2n.vlam, s=12, alpha=0.5, color='tab:blue', label=r'$v_\lambda$')
ax.scatter(c1n.vbeta, c2n.vbeta, s=12, alpha=0.5, color='tab:red', label=r'$v_\beta$')
lim = max(abs(c1n.vlam).max(), abs(c2n.vlam).max()) * 1.05
ax.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8, label='1:1')
ax.set_xlabel('case1 velocity component (deg/day)')
ax.set_ylabel('case2 velocity component (deg/day)')
ax.legend(fontsize=8); ax.grid(alpha=0.2)
ax.set_title('case1 vs case2 velocity components (same NEO, same night)\n'
             f'|Δv| median={dv12.median():.4f} deg/day')

fig.suptitle(f'Cross-case NEO velocity comparison — MJD {NIGHT} (663 common NEOs)', fontsize=11)
plt.tight_layout()
plt.savefig('Figures/tracklet_crosscase_neo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── P_NEO_vdp and P_NEO_d2 consistency across cases for common NEOs ───────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, score_col, title in [
    (axes[0], 'P_NEO_vdp', 'VDP score'),
    (axes[1], 'P_NEO_d2',  'digest2 score'),
]:
    ax.scatter(nights_neo['case1'].loc[ids, score_col],
               nights_neo['case2'].loc[ids, score_col],
               s=12, alpha=0.5, color='tab:blue', label='case1 vs case2')
    ax.scatter(nights_neo['case1'].loc[ids, score_col],
               nights_neo['case3'].loc[ids, score_col],
               s=12, alpha=0.4, color='tab:orange', marker='s', label='case1 vs case3')
    ax.plot([0,1],[0,1],'k--',lw=0.8, label='1:1')
    ax.set_xlabel(f'case1 {title}'); ax.set_ylabel(f'caseX {title}')
    ax.legend(fontsize=8); ax.grid(alpha=0.2)
    ax.set_title(f'{title} consistency across cases\n(663 common NEOs, night {NIGHT})')

plt.tight_layout()
plt.savefig('Figures/tracklet_score_consistency.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 6 — Velocity Offset from Benchmark (case1 only)

For each matched pair, compute the residual displacement:
(Δvλ, Δvβ) = (vlam_sorcha − vlam_bench, vbeta_sorcha − vbeta_bench).

The origin (0, 0) = perfect agreement between Sorcha and benchmark.

**Left panel — arrow version:** every matched pair drawn as a quiver arrow from (0,0)
to its offset.  A thick black arrow marks the median offset across all pairs.  Shows
direction and magnitude of the disagreement for each object.

**Right panel — population scatter version:** same offset points plotted as dots,
coloured by population (NEO=red, MBA=blue, TNO=gold, Trojan=green).  The median
is marked with a large ×.  Shows whether different orbital populations cluster at
different offsets — e.g. do NEOs have a systematic direction bias compared to MBAs?

In [ ]:
hit1 = matched['case1'].copy()

# ── remove false NEO matches (median + 3×MAD on NEO |Δv|) before plotting ────
neo_mask = hit1.population == 'NEO'
med_dv   = hit1.loc[neo_mask, 'dv'].median()
mad_dv   = (hit1.loc[neo_mask, 'dv'] - med_dv).abs().median()
thresh   = med_dv + 3 * mad_dv
false_match = neo_mask & (hit1.dv > thresh)
n_dropped = false_match.sum()
hit1_clean = hit1[~false_match].copy()

dvlam  = hit1_clean.vlam  - hit1_clean.vlam_b
dvbeta = hit1_clean.vbeta - hit1_clean.vbeta_b
med_dvlam  = dvlam.median()
med_dvbeta = dvbeta.median()

print(f'Dropped {n_dropped} false NEO matches (|Δv| > {thresh:.3f} deg/day)')
print(f'Plotting {len(hit1_clean)} clean pairs  |  pops: {dict(hit1_clean.population.value_counts())}')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── left: arrow version ────────────────────────────────────────────────────────
ax = axes[0]
zeros = np.zeros(len(hit1_clean))
ax.quiver(zeros, zeros, dvlam.values, dvbeta.values,
          color='steelblue', alpha=0.35, angles='xy', scale_units='xy', scale=1,
          width=0.003, label=f'Individual offsets (n={len(hit1_clean)})')
ax.quiver(0, 0, med_dvlam, med_dvbeta,
          color='black', alpha=1.0, angles='xy', scale_units='xy', scale=1,
          width=0.010, headwidth=5, headlength=5,
          label=f'Median offset\n(Δvλ={med_dvlam:.3f}, Δvβ={med_dvbeta:.3f})')
ax.scatter(0, 0, s=80, color='black', zorder=5)
ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.axvline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.set_xlabel(r'$\Delta v_\lambda$ = $v_{\lambda,\rm Sorcha} - v_{\lambda,\rm bench}$ (deg/day)')
ax.set_ylabel(r'$\Delta v_\beta$ = $v_{\beta,\rm Sorcha} - v_{\beta,\rm bench}$ (deg/day)')
ax.legend(fontsize=9, loc='upper right')
ax.set_title('Velocity offset arrows — case1 vs benchmark v3\n'
             f'(0,0) = perfect agreement  [{n_dropped} false NEO matches removed]', fontsize=10)
ax.grid(alpha=0.2)

# ── right: population scatter version ─────────────────────────────────────────
ax = axes[1]
for pop, grp in hit1_clean.groupby('population'):
    idx = grp.index
    ax.scatter(dvlam.loc[idx], dvbeta.loc[idx],
               s=30, alpha=0.7, color=POP_COLORS.get(pop, 'grey'),
               label=f'{pop} (n={len(grp)})', zorder=3)
for pop, grp in hit1_clean.groupby('population'):
    idx = grp.index
    pm_l = dvlam.loc[idx].median()
    pm_b = dvbeta.loc[idx].median()
    ax.scatter(pm_l, pm_b, s=180, marker='x', linewidths=2.5,
               color=POP_COLORS.get(pop, 'grey'), zorder=5)
ax.scatter(med_dvlam, med_dvbeta, s=350, marker='x', linewidths=3,
           color='black', zorder=6, label=f'Overall median\n({med_dvlam:.3f}, {med_dvbeta:.3f})')
ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.axvline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.set_xlabel(r'$\Delta v_\lambda$ (deg/day)')
ax.set_ylabel(r'$\Delta v_\beta$ (deg/day)')
ax.legend(fontsize=8, loc='upper right')
ax.set_title('Velocity offsets by population — case1 vs benchmark v3\n'
             'small × = per-population median,  large black × = overall median', fontsize=10)
ax.grid(alpha=0.2)

fig.suptitle(f'Sorcha − Benchmark velocity residuals  |  case1, MJD {NIGHT}  '
             f'[{n_dropped} false NEO matches removed]', fontsize=12)
plt.tight_layout()
plt.savefig('Figures/tracklet_offset_case1.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nOverall median offset: Δvλ={med_dvlam:.4f}  Δvβ={med_dvbeta:.4f} deg/day')
for pop, grp in hit1_clean.groupby('population'):
    idx = grp.index
    print(f'{pop:8s}  n={len(grp):4d}  '
          f'Δvλ med={dvlam.loc[idx].median():.3f}  '
          f'Δvβ med={dvbeta.loc[idx].median():.3f}  '
          f'|Δv| med={grp.dv.median():.3f}')

### Section 6b — NEO-only residuals: confident vs suspicious matches

The right panel above shows NEO offsets up to 2.1 deg/day — far too large to be a real
velocity difference between two pipelines measuring the same object.  These are almost
certainly **false cKDTree matches**: two different NEOs that happen to share similar (e, H).

Threshold: flag a pair as suspicious if its |Δv| exceeds **median + 3×MAD** of the NEO
|Δv| distribution (a robust outlier criterion).  Confident matches are shown in teal,
suspicious in red.  A second panel cross-plots vlam_bench vs vlam_sorcha for confident
pairs only — if matching is clean these should fall on the 1:1 line.

In [ ]:
neo1 = hit1[hit1.population == 'NEO'].copy()
neo1['dvlam']  = neo1.vlam  - neo1.vlam_b
neo1['dvbeta'] = neo1.vbeta - neo1.vbeta_b

# robust outlier threshold: median + 3×MAD
med_dv  = neo1.dv.median()
mad_dv  = (neo1.dv - med_dv).abs().median()
thresh  = med_dv + 3 * mad_dv
neo1['suspicious'] = neo1.dv > thresh

conf = neo1[~neo1.suspicious]
susp = neo1[ neo1.suspicious]
print(f'NEO pairs — total: {len(neo1)},  confident: {len(conf)},  suspicious: {len(susp)}')
print(f'Threshold |Δv| > {thresh:.3f} deg/day  (median={med_dv:.3f}, MAD={mad_dv:.3f})')
print()
if len(susp):
    print('Suspicious pairs (likely false matches):')
    display(susp[['ObjID','vlam','vbeta','vlam_b','vbeta_b','dvlam','dvbeta','dv']].sort_values('dv', ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# ── left: full NEO scatter coloured by confidence ─────────────────────────────
ax = axes[0]
ax.scatter(conf.dvlam, conf.dvbeta, s=55, alpha=0.85, color='teal',
           edgecolors='white', linewidths=0.4, label=f'Confident (n={len(conf)})', zorder=3)
ax.scatter(susp.dvlam, susp.dvbeta, s=80, alpha=0.85, color='tab:red', marker='X',
           edgecolors='darkred', linewidths=0.5, label=f'Suspicious |Δv|>{thresh:.2f} (n={len(susp)})', zorder=4)
ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.axvline(0, color='k', lw=0.6, ls='--', alpha=0.5)
# draw threshold circle
theta = np.linspace(0, 2*np.pi, 300)
ax.plot(thresh*np.cos(theta), thresh*np.sin(theta), 'k--', lw=1, alpha=0.4, label=f'|Δv|={thresh:.2f} circle')
ax.set_xlabel(r'$\Delta v_\lambda$ (deg/day)'); ax.set_ylabel(r'$\Delta v_\beta$ (deg/day)')
ax.legend(fontsize=8); ax.grid(alpha=0.2)
ax.set_title(f'All NEO offsets — case1 vs benchmark v3\n(n={len(neo1)} total)', fontsize=10)
ax.set_aspect('equal')

# ── middle: confident pairs only, zoomed ─────────────────────────────────────
ax = axes[1]
ax.scatter(conf.dvlam, conf.dvbeta, s=60, alpha=0.9, color='teal',
           edgecolors='white', linewidths=0.4, zorder=3)
# annotate median
med_cl = conf.dvlam.median(); med_cb = conf.dvbeta.median()
ax.scatter(med_cl, med_cb, s=250, marker='x', linewidths=3, color='black', zorder=5,
           label=f'Median ({med_cl:.3f}, {med_cb:.3f})')
ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.axvline(0, color='k', lw=0.6, ls='--', alpha=0.5)
ax.set_xlabel(r'$\Delta v_\lambda$ (deg/day)'); ax.set_ylabel(r'$\Delta v_\beta$ (deg/day)')
ax.legend(fontsize=8); ax.grid(alpha=0.2)
ax.set_title(f'Confident NEO pairs only (n={len(conf)})\nZoomed residuals', fontsize=10)
ax.set_aspect('equal')

# ── right: 1:1 check — vlam_bench vs vlam_sorcha for confident pairs ──────────
ax = axes[2]
ax.scatter(conf.vlam_b, conf.vlam, s=55, alpha=0.85, color='teal',
           edgecolors='white', linewidths=0.4, label=r'$v_\lambda$', zorder=3)
ax.scatter(conf.vbeta_b, conf.vbeta, s=55, alpha=0.85, color='darkorange',
           edgecolors='white', linewidths=0.4, marker='s', label=r'$v_\beta$', zorder=3)
lims = [min(conf.vlam_b.min(), conf.vlam.min(), conf.vbeta_b.min(), conf.vbeta.min()) * 1.1,
        max(conf.vlam_b.max(), conf.vlam.max(), conf.vbeta_b.max(), conf.vbeta.max()) * 1.1]
ax.plot(lims, lims, 'k--', lw=1, label='1:1')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Benchmark v3 velocity (deg/day)')
ax.set_ylabel('Sorcha case1 velocity (deg/day)')
ax.legend(fontsize=8); ax.grid(alpha=0.2); ax.set_aspect('equal')
ax.set_title(f'1:1 check — confident NEO pairs (n={len(conf)})\n'
             r'Points on dashed line = perfect agreement', fontsize=10)

fig.suptitle(f'NEO residuals: confident vs suspicious matches — case1, MJD {NIGHT}', fontsize=12)
plt.tight_layout()
plt.savefig('Figures/tracklet_neo_residuals_case1.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nConfident NEO pairs (|Δv| ≤ {thresh:.3f}):')
print(f'  Δvλ median={med_cl:.4f}  Δvβ median={med_cb:.4f}  |Δv| median={conf.dv.median():.4f} deg/day')